# 第12回：分類の評価と可視化と演習

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session12_evaluation_visualization.ipynb)


**DSゼミナールⅠ（2026年度）**
熊本大学 データサイエンス学科

今回は、分類モデルの評価指標とその可視化に絞って学びます。演習ではCSVファイルを読み込み、分類の性能を数値と図で確認します。

---

## 📋 本日の目標

1. 分類の代表的評価指標を説明できる
2. 分類可視化の目的と見方を説明できる
3. ROC曲線とPR曲線の違いを説明できる
4. CSVファイルを読み込み、分類/回帰評価と可視化を実装できる

---

## 1. 分類の評価指標

分類モデルでは、予測ラベルと実際のラベルの比較から性能を評価します。

### 1-1. 混同行列

| | 予測:陽性 | 予測:陰性 |
| ---- | ---- | ---- |
| 実際:陽性 | True Positive (TP) | False Negative (FN) |
| 実際:陰性 | False Positive (FP) | True Negative (TN) |

混同行列は、モデルの誤分類の種類を可視化する基本です。

```mermaid
flowchart LR
  A[実際:陽性] -->|TP| B[予測:陽性]
  A -->|FN| C[予測:陰性]
  D[実際:陰性] -->|FP| B
  D -->|TN| C
```

- TP：正解を正解と予測した数
- FP：誤って陽性と予測した数
- FN：誤って陰性と予測した数
- TN：正解を陰性と予測した数

### 1-2. 精度（Accuracy）

\[ \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \]

- 全体の正解率
- クラス不均衡がある場合は注意が必要

### 1-3. 適合率（Precision）

\[ \text{Precision} = \frac{TP}{TP + FP} \]

- 陽性と予測した中で正しかった割合
- 偽陽性を減らしたい場合に重要

### 1-4. 再現率（Recall）

\[ \text{Recall} = \frac{TP}{TP + FN} \]

- 実際に陽性のうち正しく検出できた割合
- 偽陰性を減らしたい場合に重要

### 1-5. F1スコア

\[ F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall} \]

- PrecisionとRecallの調和平均
- バランスの良さを測る指標

### 1-6. ROC曲線とAUC

- ROC曲線は、閾値を変えたときの真陽性率（TPR）と偽陽性率（FPR）の関係を示します
- 横軸が FPR、縦軸が TPR です
- 右上に近いほど、誤検出を少なくしつつ多くの陽性を検出できています
- AUCは曲線の下の面積で、1に近いほど性能が高いと判断します

### 1-7. PR曲線

- PR曲線は、Positiveクラスの検出精度に注目する評価です
- 横軸は Recall（実際の陽性をどれだけ拾えたか）、縦軸は Precision（予測した陽性がどれだけ正しかったか）です
- 右上に近いほど、PrecisionとRecallの両方が高い状態です
- 閾値を下げると Recall は上がりやすいが Precision は下がりやすく、閾値を上げるとその逆になります
- PR曲線は、クラスの割合が偏っている場合に特に有効です

### 1-8. ダミーデータでのROC/PRの例

以下はラベルと予測スコアの簡単なダミーデータを使って、ROC曲線とPR曲線を描く最小例です。スコアが高いほど陽性と予測されやすいと仮定します。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

np.random.seed(0)
# 50 positive, 50 negative
y_true = np.concatenate([np.ones(50), np.zeros(50)])
# スコアは陽性側が高め、負例は低めに分布させる（ノイズあり）
y_scores = np.concatenate([np.linspace(0.6, 1.0, 50), np.linspace(0.0, 0.4, 50)]) + np.random.normal(0, 0.05, 100)
y_scores = np.clip(y_scores, 0, 1)

# ROC
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curve (dummy data)')
plt.legend()
plt.grid(True)
plt.show()

# PR
precision, recall, _ = precision_recall_curve(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label=f'AP = {ap:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curve (dummy data)')
plt.legend()
plt.grid(True)
plt.show()


簡潔な読み方：

- ROC曲線は閾値を全体的に動かしたときの識別能力を表す（AUCが0.5に近いとランダム、1.0が理想）。
- PR曲線はクラス不均衡がある場合に、PrecisionとRecallのバランスを見るのに役立ちます。
- 実際のデータでは、ROC曲線とPR曲線を両方確認することで、運用に適した閾値設定を考えやすくなります。

---

## 2. 分類評価の補足

分類モデルの評価は、単に正解率を見るだけでは不十分な場合があります。以下の点を合わせて確認します。

- **クラス不均衡の影響**：少数クラスの誤分類が重要な場合、Accuracyだけで評価すると見落としがある
- **閾値による調整**：予測確率を使うモデルでは、閾値を変えることで Precision と Recall のバランスを調整できる
- **予測確率の利用**：ROC曲線やPR曲線は、閾値を変えたときの性能を評価するために使う
- **混同行列の細部確認**：TP / FP / FN / TN のバランスから、どのタイプの誤りを減らす必要があるかを判断する

### 2-1. 特徴量重要度の見方

分類モデルでは、どの説明変数がどれだけ予測に寄与しているかを確認することが大切です。モデルによって解釈の仕方が異なります。

- **決定木系モデル（Decision Tree / Random Forest など）**
  - `feature_importances_` は、特徴量を分割に使ったときの不純度減少の合計を基に算出されます
  - 値が大きいほど、その特徴量が予測に貢献した割合が高いことを示します
  - 相関の高い特徴量が複数あると分散して重要度が分かれるため、過信せずに他の指標や可視化も併せて見る必要があります

- **ロジスティック回帰**
  - 回帰係数の絶対値が大きい特徴量ほど、クラス予測に強く影響します
  - 正の係数は陽性クラスへの寄与、負の係数は陰性クラスへの寄与です
  - ただし特徴量のスケールが異なる場合は係数値を直接比較できないため、標準化や正則化を使って解釈しやすくします

- **モデルの比較としての重要度**
  - どの特徴量が重要かを確認することで、予測結果が妥当かどうかを判断できます
  - 例えば赤/白ワイン分類で `alcohol` や `volatile_acidity` が高い重要度なら、ワインの化学特性が識別に使われていることがわかります
  - 重要度だけでなく、実務上の意味やドメイン知識で解釈することが大切です

---

## 3. 演習：CSVファイルを読み込んで評価と可視化を実施しよう

### 3-1. 使用データ

以下のファイルを読み込みます。

`data/winequality.csv`

このデータは UCI Wine Quality データセットを整えたCSVです。代表的な項目は以下の通りです。

- サンプル数：およそ 6497（赤・白合わせた場合）
- 主な特徴量：`fixed acidity`, `volatile acidity`, `citric acid`, `residual sugar`, `chlorides`, `free sulfur dioxide`, `total sulfur dioxide`, `density`, `pH`, `sulphates`, `alcohol` など
- 目的変数：`type`（赤 / 白）および `quality`（整数スコア、実データではおよそ 3–9 の範囲）

今回は、分類タスクの主題として `type` を使った赤/白ワイン判別を中心に扱います。`quality >= 6` の閾値分類は補助的な例として紹介できます。

### 3-2. 演習の共通手順

以下は自力で実装する前提の基本的な流れです。

1. データ読み込み：`data/winequality.csv` を `pandas` で読み込む
2. 前処理：欠損値確認、必要なら標準化・スケーリング、文字列カテゴリの確認など
3. 特徴量選択：`fixed_acidity`, `volatile_acidity`, `citric_acid`, `residual_sugar`, `chlorides`, `free_sulfur_dioxide`, `total_sulfur_dioxide`, `density`, `ph`, `sulphates`, `alcohol` などを使う
4. 学習/評価データ分割：`train_test_split` を使い、`stratify=df['type']` などでクラスバランスを保つ
5. モデル学習と評価：分類と回帰それぞれで適切な指標を使う

### 3-3. 分類演習：赤/白ワイン判別

この演習の主題は、`type` を使った赤/白ワインの二値分類です。

- タスク設定：`y = (df['type'] == 'red').astype(int)` で赤ワインを陽性、白ワインを陰性とする
- モデル例：`LogisticRegression`, `RandomForestClassifier`, `GradientBoostingClassifier` など
- 評価指標：`Accuracy`, `Precision`, `Recall`, `F1`, `ROC AUC`, `PR AUC`
- 可視化：
  - 混同行列ヒートマップ
  - ROC曲線（`roc_curve` / `auc`）
  - PR曲線（`precision_recall_curve` / `average_precision_score`）
  - 予測確率の分布

#### Colab での日本語表示準備

Google Colab で図の日本語が文字化けしないように、以下のように `japanize-matplotlib` を使います。


In [ ]:
# Google Colab で実行する場合
!pip install -q japanize-matplotlib
import japanize_matplotlib
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "IPAexGothic"
plt.rcParams["axes.unicode_minus"] = False


通常の環境でも、`japanize_matplotlib` をインストールして同様に `plt.rcParams` を設定すれば日本語ラベルが崩れにくくなります。

#### 分類演習の実装例


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score
)

# データ読み込み
(df) = pd.read_csv('data/winequality.csv')
feature_cols = [
    'fixed_acidity','volatile_acidity','citric_acid','residual_sugar',
    'chlorides','free_sulfur_dioxide','total_sulfur_dioxide',
    'density','ph','sulphates','alcohol'
]
X = df[feature_cols]
y = (df['type'] == 'red').astype(int)


### 3-4. 回帰演習：品質スコア予測

もう一つの演習は、`quality` を回帰ターゲットにした品質スコア予測です。

- タスク設定：`y = df['quality']`
- モデル例：`LinearRegression`, `RandomForestRegressor`, `SVR`, `GradientBoostingRegressor` など
- 評価指標：`RMSE`, `MAE`, `R2`
- 可視化：
  - 実測値 vs 予測値散布図
  - 残差ヒストグラム
  - Bland-Altman プロット（実測値と予測値の差を確認）

#### 回帰演習の実装例


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# データ読み込み
(df) = pd.read_csv('data/winequality.csv')
feature_cols = [
    'fixed_acidity','volatile_acidity','citric_acid','residual_sugar',
    'chlorides','free_sulfur_dioxide','total_sulfur_dioxide',
    'density','ph','sulphates','alcohol'
]
X = df[feature_cols]
y = df['quality']


### 3-5. 実装の流れと図の例

演習では次のような流れで実装し、分類評価の図を作成します。

- データを読み込んで分布や特徴量の関係を確認する
- `type` を赤/白で二値化して分類タスクとする
- 分類モデルを学習し、混同行列やROC/PR曲線を描く
- 閾値を変えて Precision/Recall のトレードオフを確認する

実際の図の例としては、上の節にあるROC曲線とPR曲線が参考になります。回帰タスクは実測値と予測値、残差の分布を見る図を用意します。

---

## 5. 演習のポイント

- 分類の評価では、Accuracyだけでなく Precision / Recall / F1 を確認する
- ROC AUCは閾値を変えたときの性能を総合的に見る指標
- PR曲線は陽性クラスが少ないデータで特に有効
- 混同行列の誤りの種類（FP, FN）を見て、改善すべきポイントを考える
- 予測確率の閾値調整は、実務上の運用要件に合わせて行う

---

## 6. まとめ

今回の学習では、分類モデルの評価指標と可視化に重点を置きました。混同行列、ROC曲線、PR曲線を通じて、Accuracyだけでは見えないモデルの挙動を把握できるようになります。

## 7. 分類と回帰の両方の説明に使えるデータ例

- `data/heart_disease_cleveland.csv`
  - `num` を2値化して心疾患の有無を分類
  - 元の `num` をそのまま使えば疾患進行度の回帰にも使える
- `scikit-learn` の `load_diabetes()` データ
  - もともと回帰用データだが、目的変数を閾値で二値化すれば分類にも使える
- UCI `Wine Quality` データセット
  - `quality` をそのまま回帰ターゲットにする
  - `quality >= 6` などで品質分類に変換できる
- `Penguins` データセット
  - `species` による分類
  - `body_mass_g` や `bill_length_mm` を回帰ターゲットにできる

以上のデータは、分類評価と回帰評価の両方を比較しながら説明する教材として使いやすいです。
